# 01 — Exploratory Data Analysis: CSE-CIC-IDS2018

This notebook explores the [CSE-CIC-IDS2018](https://www.unb.ca/cic/datasets/ids-2018.html) dataset, which will be used to train a **Neural ODE Autoencoder** for unsupervised DDoS detection.

**Goals:**
1. **File inventory** — List all raw CSV files and their sizes.
2. **Schema consistency** — Verify that all files share the same column structure.
3. **Known data quality issues** — Detect and quantify duplicate headers, NaN/Inf values, and duplicate columns.
4. **Label distribution** — Understand the class balance between benign traffic and different attack types.
5. **Descriptive statistics** — Summarize key features relevant to the Neural ODE model.
6. **Temporal distribution** — Visualize how network flows are distributed over time.
7. **Benign vs. DDoS comparison** — Compare feature distributions between normal and attack traffic.
8. **Feature correlation** — Identify redundant or highly correlated features.
9. **Missing values audit** — Quantify NaN values per column before preprocessing.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

RAW_DIR = os.path.join("..", "data", "raw")
print(f"Looking for CSVs in: {os.path.abspath(RAW_DIR)}")

## 1. File Inventory

The CSE-CIC-IDS2018 dataset is split across multiple CSV files, each corresponding to a different day of network capture. Let's list them and check their sizes to understand the scale of the data.

In [ ]:
csv_files = sorted([f for f in os.listdir(RAW_DIR) if f.endswith(".csv")])

print(f"Total CSV files: {len(csv_files)}\n")
for f in csv_files:
    size_mb = os.path.getsize(os.path.join(RAW_DIR, f)) / 1e6
    print(f"  {f:60s} {size_mb:8.1f} MB")

## 2. Schema Consistency Check

Before concatenating all files, we verify that every CSV shares the same column structure. Schema mismatches would cause issues during preprocessing and model training.

In [ ]:
# Read the first few rows of each file to check column consistency
column_sets = {}
sample_dfs = {}

for f in csv_files:
    path = os.path.join(RAW_DIR, f)
    df_sample = pd.read_csv(path, nrows=5, encoding="utf-8", low_memory=False)
    df_sample.columns = df_sample.columns.str.strip()
    column_sets[f] = set(df_sample.columns)
    sample_dfs[f] = df_sample
    print(f"\n--- {f} ---")
    print(f"  Columns: {len(df_sample.columns)}")
    print(f"  First columns: {list(df_sample.columns[:5])}")

# Check if all files share the same columns
ref_cols = column_sets[csv_files[0]]
all_same = all(cols == ref_cols for cols in column_sets.values())
print(f"\n{'✓' if all_same else '✗'} All columns consistent across files: {all_same}")

if not all_same:
    for f, cols in column_sets.items():
        diff = cols.symmetric_difference(ref_cols)
        if diff:
            print(f"  {f}: differences = {diff}")

## 3. Full Data Load and Initial Cleaning

We load all CSV files into a single DataFrame and apply minimal cleaning so we can explore the full dataset. Heavy cleaning (imputation, normalization) is handled later in the preprocessing pipeline (`src/preprocessing.py`).

In [ ]:
dfs = []
for f in csv_files:
    path = os.path.join(RAW_DIR, f)
    df = pd.read_csv(path, encoding="utf-8", low_memory=False)
    df.columns = df.columns.str.strip()
    df["_source_file"] = f
    dfs.append(df)
    print(f"  {f}: {len(df):>10,} rows")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {len(df_all):,}")
print(f"Total columns: {len(df_all.columns)}")

## 4. Known Dataset Issues

The CSE-CIC-IDS2018 dataset has several well-documented quality issues that must be addressed before modeling:

1. **Duplicate header rows** — Some files (Feb 16, Feb 28, Mar 1) contain rows where the CSV header is repeated as data. These appear as rows where the `Label` column literally contains the string `"Label"`.
2. **Infinity and NaN values** — The `Flow Bytes/s` and `Flow Packets/s` columns contain `Inf` and `NaN` values due to zero-duration flows.
3. **Duplicate column** — `Fwd Header Length` appears twice in the schema.

We detect and quantify each issue below.

In [ ]:
# 4a. Detect rows with duplicate headers embedded as data
# These rows have the column name as value (e.g., "Label" in the Label column)
if "Label" in df_all.columns:
    header_rows = df_all[df_all["Label"] == "Label"]
    print(f"Rows with duplicate headers: {len(header_rows):,}")
    if len(header_rows) > 0:
        print("  Found in files:")
        print(header_rows["_source_file"].value_counts().to_string(header=False))
    # Remove duplicate header rows
    df_all = df_all[df_all["Label"] != "Label"].reset_index(drop=True)
    print(f"\nRows after removing duplicate headers: {len(df_all):,}")

In [ ]:
# 4b. Infinity and NaN in Flow Bytes/s and Flow Packets/s
# These arise from zero-duration flows where rate = bytes/0 = Inf
for col in ["Flow Bytes/s", "Flow Packets/s"]:
    if col in df_all.columns:
        # Convert to numeric (Infinity sometimes comes as a string)
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")
        n_inf = np.isinf(df_all[col]).sum()
        n_nan = df_all[col].isna().sum()
        print(f"{col}:")
        print(f"  Infinity: {n_inf:,}")
        print(f"  NaN:      {n_nan:,}")
        # Replace Inf with NaN for later imputation in the preprocessing pipeline
        df_all[col] = df_all[col].replace([np.inf, -np.inf], np.nan)

In [ ]:
# 4c. Duplicate column: Fwd Header Length
fwd_header_cols = [c for c in df_all.columns if "Fwd Header Length" in c]
print(f"Columns containing 'Fwd Header Length': {fwd_header_cols}")
if len(fwd_header_cols) > 1:
    print(f"  → Duplicate column detected. One copy will be removed during preprocessing.")

## 5. Label Distribution

Understanding the class balance is critical for our anomaly detection approach. Since the Neural ODE Autoencoder is trained **only on benign traffic**, we need to know:
- How much benign data is available for training.
- What types of attacks exist and their relative prevalence.
- Which files contain which attack types (the dataset is organized by attack day).

In [ ]:
# Overall label distribution
label_counts = df_all["Label"].value_counts()
label_pct = (label_counts / len(df_all) * 100).round(2)

label_summary = pd.DataFrame({"Count": label_counts, "Percentage": label_pct})
print(label_summary.to_string())

print(f"\nTotal benign: {label_counts.get('Benign', 0):,} ({label_pct.get('Benign', 0):.1f}%)")
print(f"Total attacks: {(len(df_all) - label_counts.get('Benign', 0)):,} ({100 - label_pct.get('Benign', 0):.1f}%)")

In [ ]:
# Label distribution per source file — helps understand which days contain which attacks
fig, ax = plt.subplots(figsize=(14, 6))

pivot = df_all.groupby(["_source_file", "Label"]).size().unstack(fill_value=0)
pivot.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")

ax.set_title("Label Distribution per File (Capture Day)")
ax.set_xlabel("Source File")
ax.set_ylabel("Number of Flows")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Descriptive Statistics of Key Features

We focus on features that are most relevant for the Neural ODE model, grouped by their role:

- **Temporal dynamics** — Inter-arrival times (IAT), flow rates, active/idle periods. These capture the *rhythm* of traffic, which is what the Neural ODE will learn to model.
- **Traffic structure** — Packet lengths and counts, which characterize the *shape* of each flow.
- **TCP behavior** — SYN/FIN/RST flag counts and initial window sizes, which reveal connection-level patterns that differ between normal and attack traffic.

In [ ]:
# Key features for the Neural ODE, organized by category
key_features = [
    # Temporal dynamics — these are the most important for the Neural ODE,
    # as it learns continuous-time evolution of traffic patterns
    "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Mean", "Bwd IAT Mean",
    "Flow Bytes/s", "Flow Packets/s",
    "Active Mean", "Idle Mean",
    # Traffic structure — characterizes flow shape and volume
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "Packet Length Variance",
    "Total Fwd Packets", "Total Backward Packets",
    # TCP behavior — connection-level patterns
    "SYN Flag Count", "FIN Flag Count", "RST Flag Count",
    "Init_Win_bytes_forward", "Init_Win_bytes_backward",
]

# Filter to features that actually exist in the dataset
available_features = [f for f in key_features if f in df_all.columns]
missing_features = [f for f in key_features if f not in df_all.columns]

if missing_features:
    print(f"Features not found in dataset: {missing_features}\n")

# Convert to numeric for statistical analysis
for col in available_features:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

df_all[available_features].describe().T.round(2)

## 7. Temporal Distribution of Flows

The dataset spans multiple days of network capture. Visualizing the flow count over time helps us:
- Identify periods of high vs. low activity.
- Spot attack windows (which often show traffic spikes).
- Validate that our temporal split (70/15/15) will give each split enough diversity.

In [ ]:
# Parse timestamps
if "Timestamp" in df_all.columns:
    df_all["Timestamp"] = pd.to_datetime(df_all["Timestamp"], errors="coerce", dayfirst=True)

    print(f"Time range: {df_all['Timestamp'].min()} → {df_all['Timestamp'].max()}")
    print(f"Invalid timestamps (NaT): {df_all['Timestamp'].isna().sum():,}")

    # Flows per hour
    df_valid_ts = df_all.dropna(subset=["Timestamp"])
    flows_per_hour = df_valid_ts.set_index("Timestamp").resample("1h").size()

    fig, ax = plt.subplots(figsize=(16, 5))
    flows_per_hour.plot(ax=ax)
    ax.set_title("Network Flows per Hour")
    ax.set_ylabel("Number of Flows")
    ax.set_xlabel("Time")
    plt.tight_layout()
    plt.show()
else:
    print("'Timestamp' column not found.")

## 8. Feature Distributions: Benign vs. Attack Traffic

This is the most important visualization for our project. The Neural ODE Autoencoder works by learning the *normal* distribution of traffic features — if benign and attack traffic have clearly different distributions, the model should be able to detect anomalies via high reconstruction error.

We compare histograms of key features between benign and attack flows. Features with **greater distributional separation** will contribute more to the model's discriminative power.

In [ ]:
# Create binary label column: benign (0) vs. attack (1)
df_all["is_attack"] = (df_all["Label"] != "Benign").astype(int)

# Features to compare — selected for their expected discriminative power
compare_features = [
    "Flow Packets/s", "Flow Bytes/s",
    "Flow IAT Mean", "Flow IAT Std",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "Init_Win_bytes_forward", "Packet Length Variance",
]
compare_features = [f for f in compare_features if f in df_all.columns]

n_feats = len(compare_features)
fig, axes = plt.subplots(2, (n_feats + 1) // 2, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(compare_features):
    ax = axes[i]
    benign_vals = df_all.loc[df_all["is_attack"] == 0, feat].dropna()
    attack_vals = df_all.loc[df_all["is_attack"] == 1, feat].dropna()

    # Use log scale if values span a very wide range
    use_log = benign_vals.max() > 1000 * (benign_vals.median() + 1)

    if use_log:
        benign_plot = np.log1p(benign_vals)
        attack_plot = np.log1p(attack_vals)
        ax.set_xlabel(f"log(1 + {feat})")
    else:
        benign_plot = benign_vals
        attack_plot = attack_vals
        ax.set_xlabel(feat)

    # Clip extreme outliers for cleaner visualization
    p99 = np.percentile(pd.concat([benign_plot, attack_plot]).dropna(), 99)
    bins = np.linspace(0, p99, 50)

    ax.hist(benign_plot.clip(upper=p99), bins=bins, alpha=0.5, label="Benign", density=True)
    ax.hist(attack_plot.clip(upper=p99), bins=bins, alpha=0.5, label="Attack", density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions: Benign vs. Attack Traffic", fontsize=14)
plt.tight_layout()
plt.show()

## 9. Feature Correlation Matrix

Highly correlated features carry redundant information and can slow training without improving model performance. This correlation matrix helps identify groups of correlated features that could potentially be reduced through feature selection or PCA in future iterations.

In [ ]:
# Correlation matrix of key features
corr_features = [f for f in available_features if f in df_all.columns]
corr_matrix = df_all[corr_features].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr_features)))
ax.set_yticks(range(len(corr_features)))
ax.set_xticklabels(corr_features, rotation=90, fontsize=8)
ax.set_yticklabels(corr_features, fontsize=8)
plt.colorbar(im, ax=ax, label="Pearson Correlation")
ax.set_title("Correlation Matrix of Key Features")
plt.tight_layout()
plt.show()

## 10. Missing Values Audit

A final check on data quality before preprocessing. Columns with a high percentage of missing values may need special treatment (imputation strategy, or removal if too sparse). The preprocessing pipeline uses **median imputation** for NaN values.

In [ ]:
# NaN counts per column (only showing columns that have missing values)
nan_counts = df_all.isna().sum()
nan_nonzero = nan_counts[nan_counts > 0].sort_values(ascending=False)

if len(nan_nonzero) > 0:
    print("Columns with missing values:\n")
    for col, count in nan_nonzero.items():
        pct = count / len(df_all) * 100
        print(f"  {col:40s} {count:>10,} ({pct:.2f}%)")
else:
    print("No missing values found in the dataset.")

print(f"\nTotal columns: {len(df_all.columns)}")
print(f"Numeric columns: {len(df_all.select_dtypes(include=[np.number]).columns)}")